In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from RF.rflocate import RFMapList, load_rfmap
from HD.profiles import load_hd_tc, hd_profiles
from Utils.plotting import plot_hd_tuning_curve
from Utils.statistic_utils import smooth_tuning_curve, tuning_curve_reference_radii
from HD.tuning_curve_utils import classify_hd


In [ ]:
base_dir = r"/mnt/senzailab/Kai/#Recording/m22"
date = "261008"
num_of_rec = 9
rf_session_id = 2
rf_response_window_s = (0.0, 0.2)
probe_name = "A"
hd_bins = 180
show_unit_ids = False  # Set True for an annotated view.
hd_class = 3
kappa_cutoff = 0.25
smooth_sigma_deg = 12  # Preserve the previous smoothing width in degrees.
smooth_sigma = smooth_sigma_deg / (360 / hd_bins)

session_dir = f"{base_dir}/{date}/{date}_{num_of_rec}"


In [ ]:
hd_tc = load_hd_tc(f"{session_dir}/data/tuning_curves/Probe{probe_name}/tuning_curves.tc")
classify_hd(hd_tc, kappa_cutoff=kappa_cutoff)
tc = hd_profiles(hd_tc, probe=probe_name, bins=hd_bins, hd_class=hd_class)
tc.columns = tc.columns % 360

tc_smoothed = tc.apply(lambda curve: smooth_tuning_curve(curve, sigma=smooth_sigma), axis=1)


In [ ]:
for (_, unit_id), curve in tc_smoothed.iterrows():
    result = plot_hd_tuning_curve(
        curve, int(unit_id),
        is_return_line_plot=False,
        is_return_polar_plot=True,
        clockwise=True,
        ring_radii=tuning_curve_reference_radii(curve, levels=(1,)),
    )
    plt.show()
    plt.close(result["polar_fig"])


In [ ]:
from Utils.probe_plotting import WaveformUnitPlotCollection

plot_unit_ids = tc.index.get_level_values("unit_id").tolist()
unit_plots = WaveformUnitPlotCollection(
    f"{session_dir}/data/waveform/Probe{probe_name}",
    plot_unit_ids,
    save_figures=False,
)
fig, ax = plt.subplots(figsize=(8, 12), constrained_layout=True)
position_plot = unit_plots.plot_unit_locations(show_other_units=False, ax=ax)
ax.get_legend().remove()
plt.show()
plt.close(fig)


In [ ]:
rf_session_dir = Path(base_dir) / date / f"{date}_{rf_session_id}"
rf_source = (rf_session_dir / "data/rfmapping/good/-100_400_1ms" / f"Probe{probe_name}"
             / f"regular_unitsSpikeCounts_{date}_{rf_session_id}.rfmap")
rf_maps = load_rfmap(rf_source)
hd_unit_ids = set(tc.index.get_level_values("unit_id"))
rf_maps = RFMapList([item for item in rf_maps if int(item.unit_id) in hd_unit_ids], rf_source)
rf_rate_maps = rf_maps.to_firing_rate(reconstruct_presentations=True).mean_rate(
    *rf_response_window_s, show_progress=False,
)

# Native full 2-D rate maximum, without smoothing or a detection-mask restriction.
# Ties use the first native (y, x) bin; missing/nonpositive maps retain a missing RF angle.
rf_peaks = pd.DataFrame(np.nan, index=tc.index,
                       columns=["rf_ego_deg", "rf_peak_y_deg", "rf_peak_hz", "rf_peak_ties"])
for rf_map in rf_rate_maps:
    rates = rf_map.to_2d_array()
    if not np.isfinite(rates).any() or np.nanmax(rates) <= 0:
        continue
    y, x = np.unravel_index(np.nanargmax(rates), rates.shape)
    rf_peaks.loc[(probe_name, int(rf_map.unit_id))] = [
        (rf_map.x_positions[x] + 180) % 360 - 180,
        rf_map.y_positions[y], rates[y, x], np.count_nonzero(rates == rates[y, x]),
    ]
print(f"RF peaks: {rf_peaks.rf_ego_deg.notna().sum()} / {len(tc)} HD cells; source: {rf_source}")


### HD and RF directions
Cell position is the peak of the smoothed 180-bin HD curve; the triangle tip and color give `(HD + RF ego) % 360`. RF ego retains the native 30-bin 2-D rate peak. Hollow circles have no RF peak. Exact HD ties are spaced by less than 1° for display; triangle orientations and colors use the original peaks. Color uses a fixed cyclic 0–360° scale, so similar combined directions have similar colors.


In [ ]:
# Peaks come from the smoothed 180-bin curves used in the individual panels.
preferred_direction = tc_smoothed.idxmax(axis=1).rename("hd_deg")
directions = rf_peaks.join(preferred_direction)
directions["sum_deg"] = (directions.hd_deg + directions.rf_ego_deg) % 360

# Only exact ties share a display offset, strictly inside the original 2° HD bin.
peak_groups = preferred_direction.groupby(preferred_direction)
within_bin_index = peak_groups.cumcount()
cells_per_bin = peak_groups.transform("size")
directions["display_hd_deg"] = preferred_direction + (
    (within_bin_index + 0.5) / cells_per_bin - 0.5
) * (360 / hd_bins)



In [ ]:
from matplotlib.colors import hsv_to_rgb
from matplotlib.markers import MarkerStyle
from matplotlib.path import Path as MarkerPath
from matplotlib.transforms import Affine2D

fig, ax = plt.subplots(figsize=(7.2, 7.2), subplot_kw={"projection": "polar"}, facecolor="white")
ax.set_facecolor("white")
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
circle_theta = np.linspace(0, 2 * np.pi, 361)
ax.plot(circle_theta, np.ones_like(circle_theta), color="0.82", lw=0.8, zorder=0)

# An acute tip defines one direction; the glyph centroid is the cell position.
triangle = MarkerPath([[0, 1], [-0.45, -0.5], [0.45, -0.5], [0, 1]],
                      [MarkerPath.MOVETO, MarkerPath.LINETO, MarkerPath.LINETO, MarkerPath.CLOSEPOLY])
for (_, unit_id), row in directions.iterrows():
    angle = np.deg2rad(row.display_hd_deg)
    if np.isfinite(row.sum_deg):
        # Marker rotation is in screen coordinates; north=0 and clockwise positive.
        marker = MarkerStyle(triangle, transform=Affine2D().rotate_deg(-row.sum_deg))
        ax.scatter(angle, 1, marker=marker, s=75,
                   facecolor=hsv_to_rgb([(row.sum_deg % 360) / 360, 0.75, 0.8]),
                   edgecolor="white", linewidth=0.4, zorder=3)
    else:
        ax.scatter(angle, 1, s=18, facecolor="white", edgecolor="0.55", linewidth=0.7, zorder=2)
    if show_unit_ids:
        ax.text(angle, 1.06, str(unit_id), fontsize=7, color="0.2", ha="center", va="center")

ax.set_ylim(0, 1.2)
ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
ax.set_xticklabels(["0°", "90°", "180°", "270°"], fontsize=10, color="0.2")
ax.set_yticks([])
ax.spines["polar"].set_visible(False)
ax.grid(False)
ax.set_title(f"{Path(base_dir).name} · {date} · HD {num_of_rec} / RF {rf_session_id} · {probe_name}",
             fontsize=10, color="0.25", pad=16)
fig.text(0.5, 0.045, "Position: HD    Color + tip: HD + RF ego    ○ No RF peak",
         ha="center", fontsize=9, color="0.25")
fig.subplots_adjust(left=0.09, right=0.91, bottom=0.12, top=0.88)
plt.show()
plt.close(fig)
